# Transcript Intelligence

**Goal:** Process 100 meeting transcripts from a B2B SaaS company (AegisCloud) and surface insights for product + engineering leadership.

**Approach:** All heavy lifting lives in `src/` modules. This notebook is the narrative — load, run each stage, render the results.

```
src/
├── config.py          # keyword maps, thresholds (single source of truth)
├── data_loader.py     # raw JSON → DataFrames
├── categorizer.py     # regex rules: call type, purpose, product, customer
├── sentiment.py       # meeting + sentence-level (within-call trajectories)
├── clustering.py      # TF-IDF + KMeans, k selected via silhouette
├── insights.py        # churn, incident, action items, competitive, speakers
└── visualizations.py  # all plots
```

In [ ]:
from collections import Counter

import pandas as pd
from IPython.display import display

from src import (
    categorizer,
    clustering,
    config,
    data_loader,
    insights,
    sentiment,
    visualizations,
)

pd.options.display.max_colwidth = 80

## 1. Load the dataset

Each meeting directory contains six JSON files: `meeting-info`, `transcript`, `speakers`, `speaker-meta`, `summary`, `events`. The `transcript.json` carries **per-sentence sentiment labels** that the meeting-level summary doesn't expose — we'll use that later for trajectory analysis.

In [ ]:
meetings = data_loader.load_all_meetings()
df = data_loader.meetings_to_dataframe(meetings)
speakers_df = data_loader.speakers_dataframe(meetings)
sentences_df = data_loader.sentences_dataframe(meetings)

print(f"Meetings: {len(df)}")
print(f"Sentences: {len(sentences_df):,}")
print(f"Speaker segments: {len(speakers_df):,}")
print(f"Date range: {df['start_time'].min().date()} → {df['start_time'].max().date()}")
df.head(3)

## 2. Topic & theme categorization

**Approach: hybrid (rules + clustering).** Why hybrid?

| Layer | Method | Why |
|---|---|---|
| Call type | regex on title | Titles are highly structured (`Support Case #...`, `Aegis / Customer - ...`); rules give us 100% reproducible labels |
| Meeting purpose | first-match regex (ordered) | 11 purpose categories cover the dataset cleanly |
| Product area | multi-label keyword match | A meeting can touch several products (e.g., Detect + Comply for an audit conversation) |
| Content theme | TF-IDF + KMeans | Surfaces *latent* themes that cross structural boundaries (e.g., billing conversations show up in both support and external) |

Pure-LLM would be overkill for 100 docs with this much structure — slower, costlier, less reproducible. LLMs become the right tool at 10k+ scale or for ambiguous text.

In [ ]:
# Layer 1: regex-based categorization
df = categorizer.annotate(df)
df["num_action_items"] = df["action_items"].apply(len)

print("Call types:")
print(df["call_type"].value_counts().to_string())
print("\nMeeting purposes:")
print(df["meeting_purpose"].value_counts().to_string())

product_counts = Counter(p for areas in df["product_areas"] for p in areas)
print("\nProduct mentions (multi-label):")
for p, c in product_counts.most_common():
    print(f"  {p}: {c}")

In [ ]:
# Layer 2: content clustering — k chosen by silhouette score
cluster_result = clustering.cluster_transcripts(df["full_transcript"])
df["content_cluster"] = cluster_result.labels

print(f"Optimal k = {cluster_result.n_clusters} "
      f"(silhouette = {cluster_result.silhouette:.3f})\n")
for cid, terms in cluster_result.cluster_terms.items():
    n = (df["content_cluster"] == cid).sum()
    samples = df[df["content_cluster"] == cid]["title"].head(2).tolist()
    print(f"Cluster {cid} (n={n}): {', '.join(terms)}")
    for s in samples:
        print(f"   • {s}")

In [ ]:
visualizations.plot_distribution(df)
visualizations.plot_clusters(cluster_result.tfidf_matrix, cluster_result.labels);

## 3. Sentiment analysis

We have **two sentiment signals**: meeting-level scores (1–5) from the summary, and per-sentence labels (positive/neutral/negative) from the transcript. The latter lets us compute **within-meeting trajectories** — useful for catching calls that ended fine but had a friction moment midway.

In [ ]:
df = sentiment.add_trajectories(df, sentences_df)

print("Meeting-level sentiment by call type:")
display(sentiment.summary_by_group(df, "call_type"))

print(f"\nOverall mean: {df['sentiment_score'].mean():.2f}")
print(f"Below neutral (<3.0): {(df['sentiment_score'] < 3).sum()} meetings")

In [ ]:
# By product area (multi-label, so we explode)
product_sent = (df[["sentiment_score", "product_areas"]]
                .explode("product_areas")
                .rename(columns={"product_areas": "product_area"}))

visualizations.plot_sentiment_breakdown(df, product_sent)
visualizations.plot_sentiment_trend(sentiment.weekly_trend(df))
visualizations.plot_sentiment_by_purpose(df);

**What the sentiment data is telling us:**

- **Support is most negative (mean 2.94)** — expected; customers reach out with problems.
- **Detect drives the worst product sentiment (3.20)** — directly attributable to a multi-day outage in March 2026.
- **Incident response meetings average 2.10** — by far the lowest purpose category. Organizational stress is visible in the data.
- **Weeks 10–12 show a sentiment crater across *all* call types** — the outage's blast radius is not just customer-facing.
- **Customer Onboarding (4.62) and Reviews (4.20)** are the highest-sentiment meetings — leverage these for case studies and reference building.

## 4. Strategic insights

### 4.1 Customer churn risk scoring

**Audience:** Sales leadership, CS, RevOps  
**Why it matters:** Surface at-risk accounts *before* the renewal call.

Risk score combines three normalized signals: (a) how far below-neutral the customer's average sentiment is, (b) explicit churn moments tagged in summaries, (c) sharp within-meeting sentiment drops. Weights live in `config.RISK_WEIGHTS`.

In [ ]:
health = insights.customer_health(df)
display(health.head(15))
visualizations.plot_customer_health(health);

### 4.2 Incident blast radius

**Audience:** Engineering leadership, VP CS, board  
**Why it matters:** Postmortems usually focus on the technical fix. The data shows the conversational impact extends well past the resolution window.

In [ ]:
incident = insights.incident_impact(df)
print(f"Affected meetings: {incident['n_affected']}/{incident['n_total']} "
      f"({incident['affected_pct']}%)")
print(f"  Sentiment of affected meetings: {incident['sentiment_affected']}")
print(f"  Sentiment of unaffected meetings: {incident['sentiment_unaffected']}")
print(f"  Δ = {incident['sentiment_unaffected'] - incident['sentiment_affected']:.2f}")
print("\nBy call type:")
for ct, n in incident['by_call_type'].items():
    print(f"  {ct}: {n}")

print("\nDirect incident-response timeline:")
for _, row in incident['direct_incident_meetings'].iterrows():
    print(f"  {row['start_time'].strftime('%m/%d %H:%M')} | "
          f"sent={row['sentiment_score']:.1f} | {row['title']}")

visualizations.plot_incident_blast_radius(
    incident['affected_meetings'], incident['direct_incident_meetings']);

### 4.3 Action item load distribution

**Audience:** Eng managers, PMs, ops leads  
**Why it matters:** Action items are commitments. If they cluster on a few names, those people may be execution bottlenecks; if they span call types, that person is wearing too many hats.

In [ ]:
ai_load = insights.action_item_load(df)
display(ai_load)
visualizations.plot_action_items(df, ai_load);

### 4.4 Competitive language detection

**Audience:** Product, competitive intel, sales enablement  
**Why it matters:** Customer conversations reveal what they're actually evaluating, not what analysts say.

In [ ]:
comp = insights.competitive_signals(df)
print(f"Meetings with competitive language: {comp['n_flagged']}")
print(f"  by call type: {comp['by_call_type']}")
print(f"  sentiment when flagged: {comp['sentiment_flagged']}")
print(f"  sentiment otherwise:    {comp['sentiment_other']}")

print("\nSample flagged external meetings:")
for _, row in comp['flagged_meetings'][
    comp['flagged_meetings']['call_type'] == 'external'
].head(8).iterrows():
    print(f"  {row['title']} (sentiment={row['sentiment_score']:.1f})")

### 4.5 Speaker dominance — meeting health

**Audience:** Team leads, L&D, EM  
**Why it matters:** Healthy meetings have balanced participation. Single-speaker dominance often means information asymmetry, status-update theater, or a customer not getting heard.

In [ ]:
dominance = insights.speaker_dominance(speakers_df, df)
print("Average max-speaker share by call type:")
print(dominance.groupby("call_type")["max_speaker_share"]
      .mean().apply(lambda x: f"{x:.1%}").to_string())

dominated = dominance[dominance["max_speaker_share"] > 0.6]
print(f"\nMeetings dominated by one speaker (>60%): {len(dominated)}")
for _, row in dominated.iterrows():
    print(f"  {row['max_speaker_share']:.0%} | sent={row['sentiment_score']:.1f} | {row['title']}")

visualizations.plot_speaker_dynamics(dominance);

### 4.6 Within-meeting sentiment pivots (new — uses sentence-level data)

**Audience:** CS, sales managers, coaching  
**Why it matters:** A meeting with a 3.5 average score can still contain a moment where the customer's sentiment crashed. These are *coaching gold* — exactly the moments the team should review.

We bucket each meeting's sentences into 5 equal segments, average the per-sentence sentiment, and flag meetings with the largest negative bucket-to-bucket drop.

In [ ]:
pivots = insights.negative_pivots(df)
print(f"Meetings with sharp negative pivots (max_drop ≤ -0.5): {len(pivots)}")
display(pivots.head(10))

visualizations.plot_sentiment_trajectories(df);

## 5. Summary & recommendations

### Findings at a glance
| Area | Headline |
|---|---|
| Categorization | 100 meetings → 3 call types × 11 purposes × 4 product areas; **k=7** content clusters chosen by silhouette |
| Sentiment | Support 2.94 < internal 3.42 < external 3.71. Detect product 3.20 — outage drag |
| Outage | One incident touched **68% of meetings**, dragged sentiment by ~0.8 points |
| Customer risk | Top tier flagged: Northstar Pharma, Helix Data, Quantum Edge |
| Execution | Maria Santos owns 31 action items — likely bottleneck |
| Conversation health | Support calls have ~51% single-speaker dominance — agents may be over-talking |
| Friction moments | 9 meetings with sharp within-call sentiment drops — coaching-ready |

### Recommendations
1. **This week:** Direct outreach to top-5 at-risk accounts before any renewal motion starts
2. **Next 30 days:** Redistribute action items off bottleneck individuals; add an SLA tracker for action item completion
3. **Next quarter:** Productize the churn risk score in the CRM. The signals here (sentiment delta, churn moments, within-call pivots) are derived purely from meeting transcripts — no extra instrumentation needed
4. **Long-term:** Use the incident-blast-radius framework for any future outage — measure not just MTTR but conversational recovery time

In [ ]:
# Export the processed dataset
export_cols = ["meeting_id", "title", "call_type", "meeting_purpose",
               "primary_product", "customer", "start_time", "duration_min",
               "num_participants", "sentiment_score", "overall_sentiment",
               "share_negative", "max_drop", "num_action_items", "content_cluster"]
df[export_cols].to_csv(config.OUTPUT_DIR / "meetings_processed.csv", index=False)
health.to_csv(config.OUTPUT_DIR / "customer_health.csv", index=False)
print(f"Exported to {config.OUTPUT_DIR}/")